# FTL Hackathon — FAOSTAT Wood Production: Multi-Item Analysis

**Study area:** Syrian Arab Republic  
**Period:** 2000–2024

This notebook analyzes **every wood-production item available in the FAOSTAT file** for Syria.  
For each item, it automatically:

- calculates descriptive statistics;
- calculates 2000–2024 change and CAGR when possible;
- calculates a linear trend and R²;
- compares the early and recent five-year averages;
- calculates year-on-year changes;
- generates a production trend plot;
- generates a year-on-year change plot;
- prints item-specific conclusions.

A final comparison table and comparison chart summarize all items.


In [ ]:

# ============================================================
# FTL Hackathon | FAOSTAT Wood Production
# Multi-Item Analysis — Syrian Arab Republic
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

FILE_NAME = "FAOSTAT_data_en_9-29-2026.csv"
START_YEAR = 2000
END_YEAR = 2024
TARGET_AREA = "Syrian Arab Republic"

# -----------------------------
# 1. Load data
# -----------------------------
if not os.path.exists(FILE_NAME):
    print(f"Please upload: {FILE_NAME}")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file was uploaded.")
    uploaded_filename = next(iter(uploaded))
    if uploaded_filename != FILE_NAME:
        os.rename(uploaded_filename, FILE_NAME)

df = pd.read_csv(FILE_NAME)

required = {"Area", "Item", "Year", "Value", "Unit"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# -----------------------------
# 2. Filter Syria and period
# -----------------------------
study = df[
    (df["Area"] == TARGET_AREA) &
    (df["Year"].between(START_YEAR, END_YEAR))
].copy()

if study.empty:
    candidates = [x for x in df["Area"].dropna().astype(str).unique() if "Syria" in x]
    raise ValueError(
        f"No records found for '{TARGET_AREA}'. "
        f"Syria-related Area values in the dataset: {candidates}"
    )

study["Year"] = pd.to_numeric(study["Year"], errors="coerce")
study["Value"] = pd.to_numeric(study["Value"], errors="coerce")
study = study.dropna(subset=["Year", "Value"])

items = sorted(study["Item"].dropna().unique())

print("==============================================")
print("MULTI-ITEM FAOSTAT WOOD PRODUCTION ANALYSIS")
print("==============================================")
print(f"Area: {TARGET_AREA}")
print(f"Period: {START_YEAR}–{END_YEAR}")
print(f"Number of item types: {len(items)}")
print("\nItems:")
for i, item in enumerate(items, 1):
    print(f"{i}. {item}")

# -----------------------------
# 3. Helper function
# -----------------------------
def analyze_item(item_data, item_name):
    """Calculate statistics and create two plots for one FAOSTAT item."""

    item_data = item_data.sort_values("Year").copy()

    # If duplicate records exist for a year, aggregate them transparently.
    yearly = (
        item_data.groupby(["Year", "Unit"], as_index=False)["Value"]
        .sum()
        .sort_values("Year")
    )

    unit = yearly["Unit"].iloc[0] if not yearly.empty else ""

    # Basic statistics
    avg = yearly["Value"].mean()
    maximum = yearly["Value"].max()
    minimum = yearly["Value"].min()

    max_year = int(yearly.loc[yearly["Value"].idxmax(), "Year"])
    min_year = int(yearly.loc[yearly["Value"].idxmin(), "Year"])

    # Endpoint change
    start_rows = yearly[yearly["Year"] == START_YEAR]
    end_rows = yearly[yearly["Year"] == END_YEAR]

    start_value = start_rows["Value"].iloc[0] if not start_rows.empty else np.nan
    end_value = end_rows["Value"].iloc[0] if not end_rows.empty else np.nan

    absolute_change = np.nan
    percent_change = np.nan
    cagr = np.nan

    if pd.notna(start_value) and pd.notna(end_value):
        absolute_change = end_value - start_value
        if start_value != 0:
            percent_change = absolute_change / start_value * 100
        if start_value > 0 and end_value > 0 and END_YEAR > START_YEAR:
            cagr = ((end_value / start_value) ** (1 / (END_YEAR - START_YEAR)) - 1) * 100

    # Linear trend
    if len(yearly) >= 2:
        x = yearly["Year"].to_numpy()
        y = yearly["Value"].to_numpy()
        slope, intercept = np.polyfit(x, y, 1)
        trend = slope * x + intercept
        ss_res = np.sum((y - trend) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
    else:
        slope, r2 = np.nan, np.nan
        trend = np.full(len(yearly), np.nan)

    yearly["YoY_Change_%"] = yearly["Value"].pct_change() * 100

    # Period comparison
    early = yearly[yearly["Year"].between(START_YEAR, START_YEAR + 4)]
    recent = yearly[yearly["Year"].between(END_YEAR - 4, END_YEAR)]

    early_avg = early["Value"].mean() if not early.empty else np.nan
    recent_avg = recent["Value"].mean() if not recent.empty else np.nan

    if pd.notna(early_avg) and early_avg != 0:
        period_change = (recent_avg - early_avg) / early_avg * 100
    else:
        period_change = np.nan

    # -----------------------------
    # Plot 1: production trend
    # -----------------------------
    plt.figure(figsize=(11, 5.5))
    plt.plot(
        yearly["Year"],
        yearly["Value"],
        marker="o",
        linewidth=2,
        label="Observed production"
    )

    if len(yearly) >= 2:
        plt.plot(
            yearly["Year"],
            trend,
            linestyle="--",
            linewidth=2,
            label="Linear trend"
        )

    plt.xlabel("Year")
    plt.ylabel(f"Production ({unit})")
    plt.title(f"{item_name} — Syria ({START_YEAR}–{END_YEAR})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # -----------------------------
    # Plot 2: annual change
    # -----------------------------
    yoy = yearly.dropna(subset=["YoY_Change_%"])

    plt.figure(figsize=(11, 4.8))
    plt.bar(yoy["Year"], yoy["YoY_Change_%"])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Year")
    plt.ylabel("Year-on-year change (%)")
    plt.title(f"{item_name} — Annual Percentage Change")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # -----------------------------
    # Conclusions
    # -----------------------------
    if pd.notna(start_value) and pd.notna(end_value):
        if end_value > start_value:
            direction = "increased"
        elif end_value < start_value:
            direction = "decreased"
        else:
            direction = "remained broadly unchanged"
        endpoint_sentence = (
            f"From {START_YEAR} to {END_YEAR}, production {direction} "
            f"from {start_value:,.2f} to {end_value:,.2f} {unit}."
        )
    else:
        endpoint_sentence = (
            f"Complete endpoint data for {START_YEAR} and/or {END_YEAR} "
            f"are not available for this item."
        )

    trend_sentence = (
        f"The fitted linear trend is {slope:,.2f} {unit}/year "
        f"(R²={r2:.3f})."
        if pd.notna(slope) else
        "A linear trend could not be estimated."
    )

    period_sentence = (
        f"The average changed from {early_avg:,.2f} {unit} in "
        f"{START_YEAR}–{START_YEAR+4} to {recent_avg:,.2f} {unit} in "
        f"{END_YEAR-4}–{END_YEAR} ({period_change:,.2f}%)."
        if pd.notna(early_avg) and pd.notna(recent_avg)
        else "The early/recent five-year comparison could not be calculated."
    )

    print("\n" + "-" * 65)
    print(f"CONCLUSIONS — {item_name}")
    print("-" * 65)
    print(f"1. {endpoint_sentence}")
    print(
        f"2. The highest recorded value was {maximum:,.2f} {unit} "
        f"in {max_year}; the lowest was {minimum:,.2f} {unit} in {min_year}."
    )
    print(f"3. {trend_sentence}")
    print(f"4. {period_sentence}")

    if pd.notna(cagr):
        print(f"5. CAGR over {START_YEAR}–{END_YEAR}: {cagr:,.2f}% per year.")

    # Return machine-readable summary
    return {
        "Item": item_name,
        "Unit": unit,
        "Observations": len(yearly),
        "Average": avg,
        "Minimum": minimum,
        "Minimum_Year": min_year,
        "Maximum": maximum,
        "Maximum_Year": max_year,
        f"Value_{START_YEAR}": start_value,
        f"Value_{END_YEAR}": end_value,
        "Absolute_Change": absolute_change,
        "Percent_Change": percent_change,
        "CAGR_%": cagr,
        "Linear_Trend_per_Year": slope,
        "R2": r2,
        "Early_5Y_Average": early_avg,
        "Recent_5Y_Average": recent_avg,
        "Early_to_Recent_%": period_change,
    }

# -----------------------------
# 4. Run analysis for EVERY item
# -----------------------------
all_results = []
item_tables = {}

for item in items:
    print("\n\n")
    print("=" * 80)
    print(f"ANALYZING ITEM: {item}")
    print("=" * 80)

    item_df = study[study["Item"] == item].copy()

    result = analyze_item(item_df, item)
    all_results.append(result)

    # Save yearly data for this item
    yearly_export = (
        item_df.groupby(["Year", "Unit"], as_index=False)["Value"]
        .sum()
        .sort_values("Year")
    )
    yearly_export["YoY_Change_%"] = yearly_export["Value"].pct_change() * 100
    item_tables[item] = yearly_export

# -----------------------------
# 5. Overall comparison table
# -----------------------------
summary = pd.DataFrame(all_results)

print("\n\n")
print("=" * 80)
print("OVERALL COMPARISON — ALL ITEMS")
print("=" * 80)

display(summary)

# -----------------------------
# 6. Comparison charts
# -----------------------------
comparison = summary.dropna(subset=["Percent_Change"]).copy()

if not comparison.empty:
    plt.figure(figsize=(12, 6))
    plt.barh(comparison["Item"], comparison["Percent_Change"])
    plt.axvline(0, linewidth=1)
    plt.xlabel(f"Change from {START_YEAR} to {END_YEAR} (%)")
    plt.ylabel("Wood product")
    plt.title("Change in Production by Wood Product Type")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

# -----------------------------
# 7. Export results
# -----------------------------
SUMMARY_FILE = "Syria_Wood_Production_All_Items_Summary.csv"
summary.to_csv(SUMMARY_FILE, index=False)

YEARLY_FILE = "Syria_Wood_Production_All_Items_Yearly.csv"
yearly_all = []

for item, table in item_tables.items():
    temp = table.copy()
    temp.insert(0, "Item", item)
    yearly_all.append(temp)

yearly_all = pd.concat(yearly_all, ignore_index=True)
yearly_all.to_csv(YEARLY_FILE, index=False)

print("\nExported:")
print(f"- {SUMMARY_FILE}")
print(f"- {YEARLY_FILE}")
